# Tutorial 05: Full Workflow - Research to Production

A complete end-to-end workflow showing how to take a factor from research idea through validation to production deployment.

## What You'll Learn
- Complete research workflow
- Production-ready factor implementation
- Monitoring and alerting setup
- Performance attribution
- Documentation and handoff

In [ ]:
import sys
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from typing import Dict, List
import json

sys.path.insert(0, '/home/shw/quant_projects/factor_engine')
sys.path.insert(0, '/home/shw/quant_projects/notebooks')

from utils import (
    display_success, display_warning, display_metrics,
    NotebookTimer, ProgressBar
)

print("Full Workflow Tutorial")
print(f"Date: {datetime.now().strftime('%Y-%m-%d')}")

## Phase 1: Research - Hypothesis and Initial Testing

**Hypothesis**: Stocks with rising earnings estimates relative to price tend to outperform.

We'll simulate an earnings momentum factor.

In [ ]:
from api import col, rank, ts_mean, ts_std, delay, zscore, Factor
from backend.pandas_backend import PandasBackend
from storage.datasource import DataSource
from runtime.engine import FactorEngine

# Generate synthetic market data with embedded signal
def generate_research_data(n_stocks=150, n_days=750, signal_strength=0.015):
    """Generate data with earnings momentum signal."""
    np.random.seed(42)
    
    dates = pd.date_range(end='2024-01-01', periods=n_days, freq='B')
    tickers = [f'STOCK_{i:03d}' for i in range(n_stocks)]
    
    data = []
    for ticker in tickers:
        base_price = 50 + np.random.randn() * 20
        
        # Simulate earnings estimates with momentum
        earnings_base = base_price * 0.1
        earnings_changes = np.random.randn(n_days) * 0.01
        earnings_estimates = earnings_base * (1 + np.cumsum(earnings_changes))
        
        # Prices follow earnings with signal
        prices = [base_price]
        for i in range(1, n_days):
            # Earnings momentum signal
            if i >= 60:
                eps_momentum = (earnings_estimates[i] - earnings_estimates[i-60]) / earnings_estimates[i-60]
                signal = signal_strength * eps_momentum
            else:
                signal = 0
            
            ret = np.random.randn() * 0.02 + signal
            prices.append(prices[-1] * (1 + ret))
        
        for i, date in enumerate(dates):
            data.append({
                'date': date,
                'ticker': ticker,
                'close': prices[i],
                'earnings_estimate': earnings_estimates[i],
                'volume': abs(np.random.randn() * 1e6 + 5e6),
            })
    
    df = pd.DataFrame(data).set_index(['date', 'ticker']).sort_index()
    
    # Add forward returns
    for horizon in [5, 10, 20]:
        df[f'fwd_return_{horizon}d'] = df.groupby(level='ticker')['close'].pct_change(horizon).shift(-horizon)
    
    return df

research_data = generate_research_data()
print(f"Research dataset: {len(research_data)} observations")
print(f"Date range: {research_data.index.get_level_values(0).min()} to {research_data.index.get_level_values(0).max()}")
print(f"Tickers: {research_data.index.get_level_values(1).nunique()}")

display_success("Research data generated")

## Phase 2: Factor Development

Define and test the earnings momentum factor.

In [ ]:
class PandasDataSource(DataSource):
    def __init__(self, df):
        self.df = df
    def load_column(self, name: str):
        return self.df[name]

# Create earnings momentum factor
# Signal: change in earnings estimates relative to price
earnings_momentum_expr = rank(
    (col('earnings_estimate') / delay(col('earnings_estimate'), 60) - 1) / 
    (col('close') / delay(col('close'), 60))
)

earnings_factor = Factor(
    name='earnings_momentum_60d',
    expr=earnings_momentum_expr,
    freq='1d',
    universe='equities'
)

# Calculate factor
data_source = PandasDataSource(research_data)
engine = FactorEngine(backend=PandasBackend(), data_source=data_source)

with NotebookTimer("Factor calculation"):
    result = engine.run(earnings_factor)

factor_values = result['result']

print(f"\nFactor calculated: {len(factor_values)} values")
print(f"Coverage: {factor_values.notna().sum() / len(factor_values) * 100:.1f}%")
display_success("Factor development complete")

## Phase 3: Validation - Train/Test Split

Rigorous validation with proper train/test separation.

In [ ]:
def calculate_ic_metrics(factor_values, forward_returns):
    """Calculate comprehensive IC metrics."""
    aligned = pd.DataFrame({
        'factor': factor_values,
        'returns': forward_returns
    }).dropna()
    
    if len(aligned) < 10:
        return {}
    
    # Overall IC
    ic = aligned['factor'].corr(aligned['returns'], method='spearman')
    
    # Time-series IC
    dates = aligned.index.get_level_values(0).unique()
    ic_series = []
    for date in dates:
        try:
            cross = aligned.xs(date, level=0)
            if len(cross) >= 10:
                daily_ic = cross['factor'].corr(cross['returns'], method='spearman')
                ic_series.append(daily_ic)
        except:
            pass
    
    ic_mean = np.mean(ic_series) if ic_series else np.nan
    ic_std = np.std(ic_series) if ic_series else np.nan
    ic_ir = ic_mean / ic_std if ic_std > 0 else np.nan
    
    # Win rate
    win_rate = sum(1 for ic in ic_series if ic > 0) / len(ic_series) if ic_series else np.nan
    
    return {
        'ic': ic,
        'ic_mean': ic_mean,
        'ic_std': ic_std,
        'ic_ir': ic_ir,
        'win_rate': win_rate,
        'n_periods': len(ic_series)
    }

# Split data: 60% train, 20% validation, 20% test
dates = sorted(research_data.index.get_level_values(0).unique())
n = len(dates)

train_dates = dates[:int(n*0.6)]
val_dates = dates[int(n*0.6):int(n*0.8)]
test_dates = dates[int(n*0.8):]

print(f"Train: {len(train_dates)} days ({train_dates[0]} to {train_dates[-1]})")
print(f"Val:   {len(val_dates)} days ({val_dates[0]} to {val_dates[-1]})")
print(f"Test:  {len(test_dates)} days ({test_dates[0]} to {test_dates[-1]})")

# Evaluate on each split
splits_performance = {}
for split_name, split_dates in [('train', train_dates), ('val', val_dates), ('test', test_dates)]:
    factor_split = factor_values.loc[split_dates]
    returns_split = research_data.loc[split_dates, 'fwd_return_10d']
    
    perf = calculate_ic_metrics(factor_split, returns_split)
    splits_performance[split_name] = perf
    
    print(f"\n{split_name.upper()} Performance:")
    for k, v in perf.items():
        print(f"  {k}: {v:.4f}" if not np.isnan(v) else f"  {k}: nan")

# Check for overfitting
train_ic = splits_performance['train']['ic']
test_ic = splits_performance['test']['ic']
degradation = train_ic - test_ic

print(f"\nOverfitting Check:")
print(f"  Train IC: {train_ic:.4f}")
print(f"  Test IC:  {test_ic:.4f}")
print(f"  Degradation: {degradation:.4f}")

if degradation < 0.02:
    display_success("Low overfitting risk - factor is stable")
elif degradation < 0.05:
    display_warning("Moderate overfitting detected")
else:
    display_warning("High overfitting risk - consider simplification")

## Phase 4: Production Implementation

Package the factor for production with proper documentation.

In [ ]:
# Create production-ready factor specification
production_spec = {
    'factor_name': 'earnings_momentum_60d',
    'version': '1.0.0',
    'created_date': datetime.now().isoformat(),
    'description': 'Earnings estimate momentum relative to price changes over 60 days',
    
    'parameters': {
        'lookback_period': 60,
        'frequency': '1d',
        'universe': 'equities'
    },
    
    'data_requirements': [
        'close',
        'earnings_estimate'
    ],
    
    'performance_metrics': {
        'train_ic': float(train_ic),
        'test_ic': float(test_ic),
        'train_ic_ir': float(splits_performance['train']['ic_ir']),
        'test_ic_ir': float(splits_performance['test']['ic_ir']),
        'test_win_rate': float(splits_performance['test']['win_rate']),
    },
    
    'validation': {
        'train_period': f"{train_dates[0]} to {train_dates[-1]}",
        'test_period': f"{test_dates[0]} to {test_dates[-1]}",
        'overfitting_check': 'PASS' if degradation < 0.02 else 'WARNING'
    },
    
    'monitoring_thresholds': {
        'min_ic': float(test_ic * 0.5),  # Alert if IC drops below 50% of test IC
        'min_coverage': 0.8,  # Alert if coverage < 80%
        'max_turnover': 0.3,  # Alert if daily turnover > 30%
    },
    
    'usage_notes': [
        'Requires earnings estimate data with at least 60 days history',
        'Best performance on horizon of 10-20 days',
        'Consider transaction costs due to moderate turnover',
        'Monitor earnings data quality - missing estimates will reduce coverage'
    ]
}

print("Production Specification:")
print(json.dumps(production_spec, indent=2))

display_success("Production specification created")

## Phase 5: Monitoring Setup

Define monitoring metrics and alerting logic.

In [ ]:
def monitor_factor_health(factor_values: pd.Series, forward_returns: pd.Series,
                          thresholds: Dict, lookback_days: int = 60) -> Dict:
    """Monitor factor health against defined thresholds."""
    
    # Get recent period
    recent_dates = sorted(factor_values.index.get_level_values(0).unique())[-lookback_days:]
    recent_factor = factor_values.loc[recent_dates]
    recent_returns = forward_returns.loc[recent_dates]
    
    # Calculate metrics
    recent_metrics = calculate_ic_metrics(recent_factor, recent_returns)
    coverage = recent_factor.notna().sum() / len(recent_factor)
    
    # Calculate turnover
    dates = sorted(recent_factor.index.get_level_values(0).unique())
    turnovers = []
    for i in range(1, len(dates)):
        try:
            prev_ranks = recent_factor.xs(dates[i-1], level=0).rank()
            curr_ranks = recent_factor.xs(dates[i], level=0).rank()
            common = prev_ranks.index.intersection(curr_ranks.index)
            if len(common) > 10:
                turnover = (prev_ranks[common] - curr_ranks[common]).abs().mean() / len(common)
                turnovers.append(turnover)
        except:
            pass
    
    avg_turnover = np.mean(turnovers) if turnovers else np.nan
    
    # Check thresholds
    alerts = []
    
    if recent_metrics['ic'] < thresholds['min_ic']:
        alerts.append(f"IC below threshold: {recent_metrics['ic']:.4f} < {thresholds['min_ic']:.4f}")
    
    if coverage < thresholds['min_coverage']:
        alerts.append(f"Coverage below threshold: {coverage:.2%} < {thresholds['min_coverage']:.2%}")
    
    if not np.isnan(avg_turnover) and avg_turnover > thresholds['max_turnover']:
        alerts.append(f"Turnover above threshold: {avg_turnover:.2%} > {thresholds['max_turnover']:.2%}")
    
    return {
        'status': 'HEALTHY' if not alerts else 'WARNING',
        'recent_ic': recent_metrics['ic'],
        'recent_ic_ir': recent_metrics['ic_ir'],
        'coverage': coverage,
        'avg_turnover': avg_turnover,
        'alerts': alerts,
        'lookback_days': lookback_days,
    }

# Run monitoring
monitor_result = monitor_factor_health(
    factor_values,
    research_data['fwd_return_10d'],
    production_spec['monitoring_thresholds'],
    lookback_days=60
)

print("\nFactor Health Monitor:")
print(f"Status: {monitor_result['status']}")
print(f"\nRecent Performance ({monitor_result['lookback_days']} days):")
print(f"  IC: {monitor_result['recent_ic']:.4f}")
print(f"  IC IR: {monitor_result['recent_ic_ir']:.4f}")
print(f"  Coverage: {monitor_result['coverage']:.2%}")
print(f"  Turnover: {monitor_result['avg_turnover']:.2%}")

if monitor_result['alerts']:
    print("\nAlerts:")
    for alert in monitor_result['alerts']:
        print(f"  - {alert}")
    display_warning(f"{len(monitor_result['alerts'])} alerts triggered")
else:
    display_success("Factor health check passed - no alerts")

## Phase 6: Performance Attribution

Analyze factor performance by different dimensions.

In [ ]:
def performance_attribution(factor_values: pd.Series, forward_returns: pd.Series, 
                            n_quantiles: int = 5) -> pd.DataFrame:
    """Analyze returns by factor quantile."""
    aligned = pd.DataFrame({
        'factor': factor_values,
        'returns': forward_returns
    }).dropna()
    
    # Assign quantiles
    aligned['quantile'] = pd.qcut(aligned['factor'], n_quantiles, labels=False, duplicates='drop') + 1
    
    # Calculate stats by quantile
    attribution = aligned.groupby('quantile')['returns'].agg([
        ('mean_return', 'mean'),
        ('std_return', 'std'),
        ('sharpe', lambda x: x.mean() / x.std() if x.std() > 0 else 0),
        ('count', 'count')
    ]).reset_index()
    
    attribution['annualized_return'] = attribution['mean_return'] * 252
    attribution['annualized_sharpe'] = attribution['sharpe'] * np.sqrt(252)
    
    return attribution

# Run attribution
attribution = performance_attribution(factor_values, research_data['fwd_return_10d'])

print("\nPerformance Attribution by Quantile:")
print(attribution.to_string(index=False))

# Long-short spread
q5_return = attribution[attribution['quantile'] == 5]['mean_return'].values[0]
q1_return = attribution[attribution['quantile'] == 1]['mean_return'].values[0]
spread = q5_return - q1_return
spread_annualized = spread * 252

display_metrics({
    'Q5_return': q5_return,
    'Q1_return': q1_return,
    'spread': spread,
    'spread_annualized': spread_annualized,
}, "Long-Short Performance")

if spread > 0:
    display_success(f"Positive spread: {spread_annualized:.2%} annualized")
else:
    display_warning("Negative spread detected")

## Phase 7: Documentation and Handoff

Create comprehensive documentation for production team.

In [ ]:
handoff_doc = f"""
# Factor Production Handoff: earnings_momentum_60d

## Summary
Factor measuring earnings estimate momentum relative to price changes.

## Performance Summary
- Test IC: {test_ic:.4f}
- Test IC IR: {splits_performance['test']['ic_ir']:.4f}
- Win Rate: {splits_performance['test']['win_rate']:.2%}
- Long-Short Spread: {spread_annualized:.2%} annualized

## Implementation
```python
from api import col, rank, delay, Factor

expr = rank(
    (col('earnings_estimate') / delay(col('earnings_estimate'), 60) - 1) / 
    (col('close') / delay(col('close'), 60))
)

factor = Factor('earnings_momentum_60d', expr, '1d', 'equities')
```

## Data Requirements
- `close`: Daily closing prices
- `earnings_estimate`: Analyst consensus earnings estimates
- Minimum history: 60 trading days

## Monitoring
- **IC Threshold**: {production_spec['monitoring_thresholds']['min_ic']:.4f}
- **Coverage Threshold**: {production_spec['monitoring_thresholds']['min_coverage']:.0%}
- **Turnover Threshold**: {production_spec['monitoring_thresholds']['max_turnover']:.0%}
- **Review Frequency**: Weekly

## Known Limitations
1. Requires high-quality earnings estimate data
2. Coverage may drop around earnings announcements
3. Performance may degrade in low-volatility regimes

## Contact
Research Team: factor_research@example.com
Created: {datetime.now().strftime('%Y-%m-%d')}
"""

print(handoff_doc)
display_success("Documentation complete - ready for production")

## Summary Checklist

### Research Phase ✓
- [x] Hypothesis defined
- [x] Data generated/collected
- [x] Factor implemented

### Validation Phase ✓
- [x] Train/validation/test split
- [x] Out-of-sample testing
- [x] Overfitting check
- [x] Multi-horizon evaluation

### Production Phase ✓
- [x] Production specification
- [x] Monitoring setup
- [x] Performance attribution
- [x] Documentation

## Key Takeaways

1. **Structured Workflow**: Follow a clear path from hypothesis to production
2. **Rigorous Validation**: Always use proper train/test splits and out-of-sample testing
3. **Documentation**: Document everything - assumptions, parameters, thresholds, limitations
4. **Monitoring**: Set up automated monitoring before deployment
5. **Attribution**: Understand where returns come from (quantile analysis)
6. **Handoff**: Provide complete specifications for production team

## Production Deployment Steps

1. **Code Review**: Have another researcher review the factor implementation
2. **Backtest**: Run full historical backtest with transaction costs
3. **Paper Trading**: Deploy in paper trading for 2-4 weeks
4. **Monitor**: Check daily IC, coverage, and turnover
5. **Go Live**: Gradually ramp up capital allocation
6. **Post-Deployment**: Continue monitoring and periodic review

## Next Steps

- Review example notebooks for real-world patterns
- Implement your own factors following this workflow
- Set up automated reporting and alerting
- Build a factor library with version control